# Historical feature and classifier comparison

Curated from the original collaborative PBL repository (source blob `e5daa8bd552f2579fa12ee0635edaa35df36a94e`). Original cell order and research logic are retained; local PBL paths were made relative and saved outputs/execution metadata were cleared for privacy and size. This archival notebook documents exploratory feature/classifier experiments, not a freshly validated pipeline. Original datasets and embeddings are not included. Execute individual cells only after supplying their inputs and reviewing alternative branches. Some original cells use earlier APIs or write experiment outputs; see [data preparation notes](../../docs/data_preparation.md).


In [ ]:
import sys

sys.path.append('..')

In [ ]:
import numpy as np
from src.preprocessing import batch_encode_and_flatten
import pandas as pd
from pathlib import Path
from src.utils import calculate_search_grid

# Import dataset / sequences
df_path = Path('../data/preprocessed/exp_proteins_c100_flens25-75_usamp_enz.tsv')

df = pd.read_csv(df_path, sep='\t')

In [ ]:
from protlearn import features
from sklearn.preprocessing import StandardScaler
# Get and encode the sequences using AA composition
seqs = list(df['Sequence'])
labels = list(df['Enzyme'])
aaindex_encodings = features.aaindex1(seqs)[0]
digram_encodings = features.ngram(seqs, n=2)[0]
encodings_hybrid = np.array([np.concat((digram_encodings[i], aaindex_encodings[i])) for i in range(0, len(aaindex_encodings))])
scaler = StandardScaler()
scaled_encodings_hybrid = scaler.fit_transform(encodings_hybrid)
X = scaled_encodings_hybrid
y = labels

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix

# Using simple train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=45)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC

# Initializing the estimators
rf = RandomForestClassifier()
logr = LogisticRegression(max_iter=1000)
svm = LinearSVC()

In [ ]:
# Random Forest
n_iter = 10
param_dist_rf = {'max_depth': [None, 10, 20,], 'max_features': ['sqrt', 'log2'], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2, 4]}
rand_grid_rf = calculate_search_grid(rf, X_train, y_train, X_test, y_test, n_iter, param_dist_rf)
rand_grid_rf.to_csv('../results/performance_eval/gridsearch_model_comparison/hybrid_aaindex_ngrams/hybrid_aaindex_ngrams_rf.tsv', sep='\t', index=False)

In [ ]:
# Logistic Regression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
n_iter = 10
param_dist_logr = {'max_iter': [1000], 'C': [0.1, 1, 10, 100, 1000], 'tol': [0.00001, 0.0001, 0.001, 0.01]}
rand_grid_logr = calculate_search_grid(logr, X_train, y_train, X_test, y_test, n_iter, param_dist_logr, randomized=False)
rand_grid_logr.to_csv('../results/performance_eval/gridsearch_model_comparison/hybrid_aaindex_ngrams/hybrid_aaindex_ngrams_logr.tsv', sep='\t', index=False)

In [ ]:
# SVM
n_iter = 10
param_dist_svm = {'max_iter': [1000], 'loss': ['hinge', 'squared_hinge'], 'C': [1, 10, 100], 'fit_intercept': [True, False]}
rand_grid_svm = calculate_search_grid(svm, X_train, y_train, X_test, y_test, n_iter, param_dist_svm, randomized=True)
rand_grid_svm.to_csv('../results/performance_eval/gridsearch_model_comparison/hybrid_aaindex_ngrams/hybrid_aaindex_ngrams_svm.tsv', sep='\t', index=False)